In [ ]:
# install needed libraries
#!pip install fastopic
#!pip install topmost
#!pip install -U kaleido

# resources and tutorials
# https://colab.research.google.com/drive/1_b55QpVQGFBX9PsyrYfNxDzbJ1IjrUdH?usp=sharing#scrollTo=EDKpq0PqHxOL
# https://github.com/BobXWu/FASTopic?tab=readme-ov-file#apis
# https://github.com/BobXWu/FASTopic/blob/master/fastopic/_utils.py
# https://huggingface.co/blog/bobxwu/fastopic
# https://arxiv.org/pdf/2405.17978.pdf - NeurIPS 2024 paper


In [ ]:
# load libraries
import pandas as pd
from fastopic import FASTopic
import topmost as topmost
from topmost.preprocessing import Preprocessing
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import plotly.io as pio
import kaleido

In [ ]:
# load data
data = pd.read_csv('data/df_out_all.csv')

In [ ]:
# keep only rows with "selected_TM" TRUE
data = data[data['selected_TM'] == True]

In [ ]:
# paste titles, keywords and abstract in text variable
data['text'] = data['dc:title'] + ' ' + data['Index Keywords'] + ' ' + data['Abstract']

In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    if not isinstance(text, str):  # Check if input is a string
        return ''
    words = text.split()  # Use split instead of word_tokenize
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return ' '.join(filtered_words)  # Return cleaned text as a single string

# Apply the function to your 'text_preprocessed' column
data['text_preprocessed'] = data['text'].apply(lambda x: remove_stopwords(x))
#data['text_preprocessed'] = data['text'].apply(remove_stopwords)
data = data.reset_index(drop=True) # index became non sequential because of removing rows in preprocessing, reset it
print(data['text_preprocessed'].head())
data.columns = data.columns.str.strip()


In [ ]:
preprocessing = Preprocessing(vocab_size=1494) # actual different tokens in data (taken from FASTtopic fitting verbose output
#model = FASTopic(50, preprocessing, doc_embed_model='all-mpnet-base-v2', verbose=True,normalize_embeddings=True) # input here any model from https://www.sbert.net/docs/sentence_transformer/pretrained_models.html
model = FASTopic(10, preprocessing, doc_embed_model='allenai-specter', verbose=True,normalize_embeddings=True, # input here any model from https://www.sbert.net/docs/sentence_transformer/pretrained_models.html
                 epochs=100,batch_size=len(data['text_preprocessed']),DT_alpha=0.5) # DT_alpha is the Dirichlet prior for the document-topic distribution, higher, more discrimination
topic_top_words, doc_topic_dist = model.fit_transform(data['text_preprocessed'])

In [ ]:
# evaluate topic diversity (metric of goodness of fit)
TD = topmost.evaluations.compute_topic_diversity(topic_top_words)

In [ ]:
# which are the most prevalent topics across documents?
# convert to pandas format
doc_topic_dist_df = pd.DataFrame(doc_topic_dist)
# get the most prevalent topic for each document
doc_topic_dist_df['top_1'] = doc_topic_dist_df.idxmax(axis=1)
# count the number of documents per topic
doc_topic_dist_df['top_1'].value_counts()

In [ ]:
topic_top_words_df = pd.DataFrame(topic_top_words)
# split words in column 0 to have one word per column
topic_top_words_df = topic_top_words_df[0].str.split(' ', expand=True)

In [ ]:
fig = model.visualize_topic_hierarchy()
fig.show()
pio.write_image(fig, "Fastopic/Fastopic_final_10_100/topic_hierarchy.svg", format="svg")

In [ ]:
# save outputs

doc_topic_dist_df.to_csv('out/doc_topic_dist_annotated.csv', index=True, header=True)

# save model as recomended by Fastopic
model.save("out/fastopic_model")

# save model as joblib
import torch
type(model)
import joblib
# Save the model to a file
joblib.dump(model, "out/fastopic_model.joblib")